# DL GenAI Project: Milestone 1 and Milestone 2

**Learner:** Sripathi Dhruv Reddy (`23f3001736`)  
**Competition:** SurfaceGuard AI: Industrial Surface Defect Detection

This notebook follows the Milestone 1 guideline: combination-stratified 80/20
split, a native-resolution segmentation baseline trained for five epochs, a
threshold sweep at 0.3/0.5/0.7/0.9 without area filtering, and a real test
submission generated from the trained checkpoint.

In [ ]:
from pathlib import Path
import gc, json, os, random

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 1736
H, W = 128, 800
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
THRESHOLDS = [0.3, 0.5, 0.7, 0.9]
EPOCHS = 5
BATCH_SIZE = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## Locate the official competition files

In [ ]:
roots = list(Path("/kaggle/input").glob("**/public/train.csv"))
if not roots:
    roots = list(Path("/kaggle/input").glob("**/train.csv"))
assert roots, "Official train.csv was not found. Attach the competition data source."
TRAIN_CSV = roots[0]
DATA_ROOT = TRAIN_CSV.parent
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
print({"train_csv": str(TRAIN_CSV), "train_dir": str(TRAIN_DIR), "test_dir": str(TEST_DIR)})

train_long = pd.read_csv(TRAIN_CSV, keep_default_na=False)
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
assert list(train_long.columns) == ["image_id", "defect_type", "mask_rle", "width", "height"]
assert list(sample.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
print("train rows/images:", len(train_long), train_long.image_id.nunique())
print("test rows/images:", len(sample), sample.image_id.nunique())

## RLE and combination-stratified 80/20 split

Competition RLE uses 1-based starts and column-major (Fortran) flattening.

In [ ]:
def rle_decode(rle, shape=(H, W)):
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    text = str(rle).strip()
    if text:
        values = np.asarray(text.split(), dtype=np.int64).reshape(-1, 2)
        for start, length in values:
            mask[start - 1:start - 1 + length] = 1
    return mask.reshape(shape, order="F")


def rle_encode(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    padded = np.concatenate([[0], pixels, [0]])
    changes = np.where(padded[1:] != padded[:-1])[0] + 1
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes))


wide = train_long.pivot(index="image_id", columns="defect_type", values="mask_rle").fillna("")
wide = wide.reindex(columns=CLASSES, fill_value="")
present = wide.applymap(lambda x: bool(str(x).strip()))
combo = present.astype(int).astype(str).agg("".join, axis=1)
combo_counts = combo.value_counts()
# Singleton combinations cannot be stratified; group only those as "rare".
strata = combo.where(combo.map(combo_counts) >= 2, "rare")
train_ids, val_ids = train_test_split(
    wide.index.to_numpy(), test_size=0.20, random_state=SEED, stratify=strata
)
print("split:", len(train_ids), len(val_ids))
print("combination counts:\n", combo.value_counts().sort_index())

In [ ]:
def find_image(folder, image_id):
    path = folder / f"{image_id}.jpg"
    if path.exists():
        return path
    matches = list(folder.glob(f"{image_id}.*"))
    if not matches:
        raise FileNotFoundError(image_id)
    return matches[0]


class SurfaceDataset(Dataset):
    def __init__(self, image_ids, masks=None, augment=False):
        self.ids = list(image_ids)
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        folder = TRAIN_DIR if self.masks is not None else TEST_DIR
        image = cv2.imread(str(find_image(folder, image_id)), cv2.IMREAD_GRAYSCALE)
        assert image is not None and image.shape == (H, W), (image_id, None if image is None else image.shape)
        image = image.astype(np.float32) / 255.0
        image = (image - 0.5) / 0.25
        if self.masks is None:
            return torch.from_numpy(image[None]), image_id
        mask = np.stack([rle_decode(self.masks.loc[image_id, c]) for c in CLASSES]).astype(np.float32)
        if self.augment and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, :, ::-1])
        return torch.from_numpy(image[None]), torch.from_numpy(mask), image_id


train_loader = DataLoader(SurfaceDataset(train_ids, wide, True), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(SurfaceDataset(val_ids, wide, False), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)

## Native-resolution compact U-Net baseline

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(),
        )
    def forward(self, x): return self.net(x)


class TinyUNet(nn.Module):
    def __init__(self, base=8):
        super().__init__()
        self.e1, self.e2, self.e3 = ConvBlock(1, base), ConvBlock(base, base*2), ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, len(CLASSES), 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(x), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)


def loss_fn(logits, target):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + (1 - dice.mean())


model = TinyUNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for images, masks, _ in train_loader:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = loss_fn(model(images), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item()
    mean_loss = running / len(train_loader)
    history.append({"epoch": epoch, "train_loss": mean_loss})
    print(f"epoch {epoch}/{EPOCHS} train_loss={mean_loss:.5f}")

torch.save(model.state_dict(), "/kaggle/working/milestone1_tiny_unet.pt")
pd.DataFrame(history).to_csv("/kaggle/working/training_history.csv", index=False)

## Holdout threshold sweep (no minimum-area filtering)

For each threshold I check: (a) how often an empty image/class pair gets a wrong mask,
(b) Dice on pairs that really have a defect, (c) the plain mean Dice over all pairs, and (d) the
competition style score, which also gives marks for getting empty masks right.

In [ ]:
def collect_validation():
    model.eval(); probs, truths = [], []
    with torch.no_grad():
        for images, masks, _ in val_loader:
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                p = torch.sigmoid(model(images.to(DEVICE, non_blocking=True)))
            probs.append(p.float().cpu().numpy().astype(np.float16))
            truths.append(masks.numpy().astype(np.uint8))
    return np.concatenate(probs), np.concatenate(truths)


val_prob, val_true = collect_validation()
true_positive = val_true.sum(axis=(2, 3)) > 0
empty_pair = ~true_positive
rows = []
for threshold in THRESHOLDS:
    pred = val_prob >= threshold
    pred_positive = pred.sum(axis=(2, 3)) > 0
    intersection = (pred & (val_true > 0)).sum(axis=(2, 3))
    denominator = pred.sum(axis=(2, 3)) + val_true.sum(axis=(2, 3))
    pair_dice = np.where(denominator > 0, 2 * intersection / np.maximum(denominator, 1), 1.0)
    fpr_empty = pred_positive[empty_pair].mean()
    fp_pixel_fraction = pred[empty_pair].mean()
    false_positive_score = 1.0 - fp_pixel_fraction
    empty_accuracy = 1.0 - fpr_empty
    foreground_dice = pair_dice[true_positive].mean()
    naive_dice = pair_dice.mean()
    # Official metric: 70% positive-pair Dice, 20% FP-pixel score, 10% empty accuracy.
    official_score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    rows.append(dict(threshold=threshold, empty_false_positive_rate=fpr_empty,
                     foreground_dice=foreground_dice, naive_mean_dice=naive_dice,
                     fp_pixel_fraction=fp_pixel_fraction,
                     false_positive_score=false_positive_score,
                     empty_accuracy=empty_accuracy, official_score=official_score))

results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/threshold_sweep.csv", index=False)
print(results.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
best_official = float(results.loc[results.official_score.idxmax(), "threshold"])
best_naive = float(results.loc[results.naive_mean_dice.idxmax(), "threshold"])
print("best official threshold:", best_official, "best naive threshold:", best_naive)

first, last = results.iloc[0], results.iloc[-1]
q12 = (
    f"Across tau=0.3 to 0.9, the empty-pair false-positive rate changed from "
    f"{first.empty_false_positive_rate:.4f} to {last.empty_false_positive_rate:.4f}, while positive-pair "
    f"foreground Dice changed from {first.foreground_dice:.4f} to {last.foreground_dice:.4f}. "
    f"The official-style score peaked at tau={best_official:.1f}; naive mean Dice peaked at tau={best_naive:.1f}. "
    "Higher thresholds suppress spurious masks on the many empty image-class pairs, so the naive mean can "
    "prefer a higher tau because true-negative empty pairs dominate it. The official score gives 70% weight "
    "to foreground Dice, 20% to the false-positive pixel score, and 10% to empty accuracy, so it limits that "
    "empty-pair advantage and gives a fairer choice."
)
Path("/kaggle/working/milestone1_q12_answer.txt").write_text(q12)
print("\nQ12 answer:\n", q12)

del val_prob, val_true
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## Generate the real competition submission

In [ ]:
test_ids = sample.image_id.drop_duplicates().tolist()
test_loader = DataLoader(SurfaceDataset(test_ids), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)
prediction_by_key = {}
model.eval()
with torch.no_grad():
    for images, image_ids in test_loader:
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            probs = torch.sigmoid(model(images.to(DEVICE, non_blocking=True))).float().cpu().numpy()
        for i, image_id in enumerate(image_ids):
            for class_index, defect_type in enumerate(CLASSES):
                prediction_by_key[(image_id, defect_type)] = rle_encode(probs[i, class_index] >= best_official)

submission = sample.copy()
submission["mask_rle"] = [prediction_by_key[(i, c)] for i, c in zip(submission.image_id, submission.defect_type)]
# Kaggle's validator expects a literal space for an empty mask, matching sample_submission.csv.
submission["mask_rle"] = submission["mask_rle"].replace("", " ")
assert list(submission.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
assert len(submission) == len(sample) and submission.ID.equals(sample.ID)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("submission rows:", len(submission), "non-empty masks:", (submission.mask_rle.str.len() > 0).sum())
print("Saved /kaggle/working/submission.csv")

---
# Milestone 2: Segmentation pipeline and first improved submission

Everything below reuses the helpers defined above (`rle_decode`, `rle_encode`, `CLASSES`, `TRAIN_DIR`, `wide`).
Competition RLE is 1-based and column-major (`order='F'`).

## Q1. Column-major RLE on the 8 x 8 grid

In [ ]:
# highlighted cells in the question figure (1-based, column-major numbering)
active = [12, 13, 23, 24, 25, 26, 44, 45, 47]
grid = np.zeros((8, 8), dtype=np.uint8)
for idx in active:
    i = idx - 1
    grid[i % 8, i // 8] = 1          # row = i % H, col = i // H
print(grid)
q1 = rle_encode(grid)
print("Q1 RLE:", q1)
assert (rle_decode(q1, (8, 8)) == grid).all()   # round trip check

## Q2. Decoding all four masks of image 0ZUU8LH57Y

In [ ]:
rows = train_long[train_long.image_id == "0ZUU8LH57Y"]
per_class = {r.defect_type: int(rle_decode(r.mask_rle).sum()) for r in rows.itertuples()}
print(per_class)
print("Q2 total positive pixels:", sum(per_class.values()))

## Q3. Class distribution and prevalence

In [ ]:
present_wide = wide.apply(lambda col: col.str.strip() != "").astype(int)[CLASSES]
counts = present_wide.sum()
summary = pd.DataFrame({"images": counts, "prevalence_%": (100 * counts / len(present_wide)).round(2)})
print("rows:", len(train_long), "images:", len(present_wide))
print(summary)
print("clean images (no defect):", int((present_wide.sum(1) == 0).sum()))
print("Q3:", f"{counts.idxmax()} ({counts.max()}), {counts.idxmin()} ({counts.min()})")

## Q4. Co-occurrence matrix C = X^T X

In [ ]:
X = present_wide.values
C = pd.DataFrame(X.T @ X, index=CLASSES, columns=CLASSES)
print(C)
off = C.where(~np.eye(4, dtype=bool)).stack()
pair = off.idxmax()
print("Q4:", pair, int(off.max()))

## Q5. Co-occurrence vs correlation (phi coefficient)

In [ ]:
corr = present_wide.corr()
print(corr.round(4))
n = len(present_wide)
for i in range(4):
    for j in range(i + 1, 4):
        a, b = CLASSES[i], CLASSES[j]
        expected = counts[a] * counts[b] / n
        print(f"{a}-{b}: observed {C.loc[a, b]}, expected if independent {expected:.1f}, phi {corr.loc[a, b]:.4f}")
print("Q5:", round(corr.loc[pair[0], pair[1]], 2))

## Q6. Pixel-level overlap between defect classes

In [ ]:
multi_ids = present_wide.index[present_wide.sum(1) >= 2]
max_overlap, max_dice, checked = 0, 0.0, 0
for image_id in multi_ids:
    masks_here = [rle_decode(wide.loc[image_id, c]) for c in CLASSES if present_wide.loc[image_id, c]]
    for i in range(len(masks_here)):
        for j in range(i + 1, len(masks_here)):
            inter = int((masks_here[i] & masks_here[j]).sum())
            dice = 2 * inter / (masks_here[i].sum() + masks_here[j].sum())
            max_overlap, max_dice, checked = max(max_overlap, inter), max(max_dice, dice), checked + 1
print("multi-label images:", len(multi_ids), "pairs checked:", checked)
print("Q6 max overlap pixels:", max_overlap, "max pairwise Dice:", max_dice)

## Q7. PyTorch Dataset for Paradigm 1 (RGB image, 4-channel float32 mask)

`__getitem__` returns a tuple `(image, masks)`: image `[3, 128, 800]` in [0, 1], masks `[4, 128, 800]` float32.

In [ ]:
class SurfaceAnomalyDataset(Dataset):
    def __init__(self, image_df, image_dir, augment=False):
        self.df = image_df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(str(self.image_dir / f"{row['image_id']}.jpg"), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        masks = np.stack([rle_decode(row[c]) for c in CLASSES]).astype(np.float32)
        image = image.transpose(2, 0, 1)
        if self.augment:
            if random.random() < 0.5:
                image, masks = image[:, :, ::-1], masks[:, :, ::-1]
            if random.random() < 0.5:
                image, masks = image[:, ::-1, :], masks[:, ::-1, :]
        image = torch.from_numpy(np.ascontiguousarray(image))
        masks = torch.from_numpy(np.ascontiguousarray(masks))
        return image, masks


wide_df = wide.reset_index()
check_ds = SurfaceAnomalyDataset(wide_df, TRAIN_DIR)
images, masks = next(iter(DataLoader(check_ds, batch_size=4, shuffle=False)))
print("dataset length:", len(check_ds))
print("images:", list(images.shape), images.dtype, "range", float(images.min()), float(images.max()))
print("masks:", list(masks.shape), masks.dtype)

## Q8. Baseline 3-stage U-Net, 5 epochs, AdamW lr 3e-4, BCEWithLogitsLoss

In [ ]:
BASE_EPOCHS = 5
tr_df = wide_df[wide_df.image_id.isin(train_ids)]
va_df = wide_df[wide_df.image_id.isin(val_ids)]
base_train_loader = DataLoader(SurfaceAnomalyDataset(tr_df, TRAIN_DIR), batch_size=16, shuffle=True,
                               num_workers=4, pin_memory=True, drop_last=True)
base_val_loader = DataLoader(SurfaceAnomalyDataset(va_df, TRAIN_DIR), batch_size=16, shuffle=False,
                             num_workers=4, pin_memory=True)


def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class UNet3(nn.Module):
    def __init__(self, b=16):
        super().__init__()
        self.e1, self.e2, self.e3 = block(3, b), block(b, 2 * b), block(2 * b, 4 * b)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(4 * b, 8 * b)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2), block(8 * b, 4 * b)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), block(4 * b, 2 * b)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * b, b, 2, 2), block(2 * b, b)
        self.out = nn.Conv2d(b, 4, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        m = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(m), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)


torch.manual_seed(SEED)
base_model = UNet3().to(DEVICE)
base_opt = torch.optim.AdamW(base_model.parameters(), lr=3e-4)
criterion = nn.BCEWithLogitsLoss()
base_history = []
for epoch in range(1, BASE_EPOCHS + 1):
    base_model.train(); total, batches = 0.0, 0
    for x, y in base_train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        base_opt.zero_grad(set_to_none=True)
        loss = criterion(base_model(x), y)
        loss.backward(); base_opt.step()
        total += loss.item(); batches += 1
    base_model.eval(); vtotal, vcount = 0.0, 0
    with torch.no_grad():
        for x, y in base_val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            vtotal += criterion(base_model(x), y).item() * x.size(0); vcount += x.size(0)
    base_history.append({"epoch": epoch, "train_loss": total / batches, "val_loss": vtotal / vcount})
    print(f"epoch {epoch}: train_loss={total / batches:.4f} val_loss={vtotal / vcount:.4f}")
    if base_history[-1]["val_loss"] == min(h["val_loss"] for h in base_history):
        torch.save(base_model.state_dict(), "/kaggle/working/m2_baseline_best.pt")

base_history = pd.DataFrame(base_history)
base_history.to_csv("/kaggle/working/m2_baseline_history.csv", index=False)
print(base_history)
print("best epoch:", int(base_history.loc[base_history.val_loss.idxmin(), "epoch"]))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.plot(base_history.epoch, base_history.train_loss, marker="o", label="train")
plt.plot(base_history.epoch, base_history.val_loss, marker="o", label="validation")
plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.title("Q8 baseline U-Net loss"); plt.legend()
plt.tight_layout(); plt.show()

---
# Milestone 3: three models compared with Weights & Biases

The project needs at least three models and a W&B run for each one:
- **Model A, built from scratch:** my own 3 stage U-Net (wider version of the Q8 baseline)
- **Model B, pretrained:** U-Net with an ImageNet pretrained ResNet34 encoder
- **Model C, my choice:** FPN decoder with an ImageNet pretrained EfficientNet-B2 encoder

All three use the same split, loss (BCE + Dice), flips as augmentation and the same validation metrics,
so the W&B runs in project `23f3001736-t32026` can be compared directly.

In [ ]:
import os, subprocess, sys, time
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
    import segmentation_models_pytorch as smp
import wandb
from sklearn.metrics import f1_score

WANDB_PROJECT = "23f3001736-t32026"
os.environ["WANDB_DIR"] = "/kaggle/working"
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB_MODE = "online"
except Exception as err:
    # no key here: runs are saved offline in /kaggle/working/wandb and uploaded later with `wandb sync`
    print("W&B key not found, logging offline:", err)
    WANDB_MODE = "offline"
try:
    # my W&B team (project 23f3001736-t32026 lives there)
    WANDB_ENTITY = UserSecretsClient().get_secret("WANDB_ENTITY")
except Exception:
    WANDB_ENTITY = "23f3001736-indian-institute-of-technology-madras"
print("W&B mode:", WANDB_MODE, "| entity:", WANDB_ENTITY)

A_EPOCHS, B_EPOCHS, C_EPOCHS = 15, 25, 25
amp_on = DEVICE.type == "cuda"
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Normalized(nn.Module):
    # the pretrained encoders expect ImageNet normalised input
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("mean", MEAN.clone())
        self.register_buffer("std", STD.clone())

    def forward(self, x):
        return self.net((x - self.mean) / self.std)


def bce_dice(logits, target):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + (1 - dice.mean())


m3_train_loader = DataLoader(SurfaceAnomalyDataset(tr_df, TRAIN_DIR, augment=True), batch_size=16, shuffle=True,
                             num_workers=4, pin_memory=True, drop_last=True)
m3_val_loader = DataLoader(SurfaceAnomalyDataset(va_df, TRAIN_DIR), batch_size=16, shuffle=False,
                           num_workers=4, pin_memory=True)

In [ ]:
def evaluate(model, loader):
    # val loss, Dice on pairs with a defect, accuracy and macro F1 of 'is this defect present'
    model.eval()
    total, count, inter, pred_px, gt_px = 0.0, 0, [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=amp_on):
                logits = model(x).float()
            total += bce_dice(logits, y).item() * x.size(0)
            count += x.size(0)
            pred = torch.sigmoid(logits) > 0.5
            inter.append((pred & (y > 0.5)).sum((2, 3)).cpu())
            pred_px.append(pred.sum((2, 3)).cpu())
            gt_px.append(y.sum((2, 3)).cpu())
    inter, pred_px, gt_px = torch.cat(inter).numpy(), torch.cat(pred_px).numpy(), torch.cat(gt_px).numpy()
    has_defect, said_defect = gt_px > 0, pred_px > 0
    dice = 2 * inter / np.maximum(pred_px + gt_px, 1)
    return {
        "val_loss": total / count,
        "val_dice": float(dice[has_defect].mean()),
        "val_accuracy": float((has_defect == said_defect).mean()),
        "val_f1": float(np.mean([f1_score(has_defect[:, c], said_defect[:, c], zero_division=0) for c in range(4)])),
    }


def train_model(name, model, epochs, lr, notes):
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name, mode=WANDB_MODE, reinit=True,
                     config={"model": name, "epochs": epochs, "lr": lr, "batch_size": 16, "loss": "bce+dice",
                             "augment": "hflip+vflip", "notes": notes})
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(m3_train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on)
    best, best_path, history = -1.0, f"/kaggle/working/m3_{name}.pt", []
    for epoch in range(1, epochs + 1):
        start = time.time()
        model.train()
        running, batches = 0.0, 0
        for x, y in m3_train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp_on):
                loss = bce_dice(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
            batches += 1
        metrics = {"epoch": epoch, "train_loss": running / batches, **evaluate(model, m3_val_loader),
                   "lr": sched.get_last_lr()[0]}
        history.append(metrics)
        wandb.log(metrics)
        print(f"{name} epoch {epoch}: train {metrics['train_loss']:.4f} val {metrics['val_loss']:.4f} "
              f"dice {metrics['val_dice']:.4f} acc {metrics['val_accuracy']:.4f} f1 {metrics['val_f1']:.4f} "
              f"({time.time() - start:.0f}s)", flush=True)
        if metrics["val_dice"] > best:
            best = metrics["val_dice"]
            torch.save(model.state_dict(), best_path)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    best_row = max(history, key=lambda h: h["val_dice"])
    for k, v in best_row.items():
        run.summary["best_" + k] = v
    run.finish()
    pd.DataFrame(history).to_csv(f"/kaggle/working/m3_{name}_history.csv", index=False)
    return model, best_row

### Model A: U-Net built from scratch

In [ ]:
torch.manual_seed(SEED)
model_a, best_a = train_model("A_scratch_unet", Normalized(UNet3(b=32)), A_EPOCHS, 1e-3,
                              "3 stage U-Net, random init, no pretrained weights")

### Model B: pretrained ResNet34 U-Net

In [ ]:
torch.manual_seed(SEED)
model_b, best_b = train_model("B_resnet34_unet",
                              Normalized(smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=4)),
                              B_EPOCHS, 3e-4, "U-Net decoder, ImageNet ResNet34 encoder")

### Model C: FPN with a pretrained EfficientNet-B2 encoder

In [ ]:
torch.manual_seed(SEED)
try:
    net_c, c_name = smp.FPN("efficientnet-b2", encoder_weights="imagenet", in_channels=3, classes=4), "C_effb2_fpn"
except Exception as err:
    print("EfficientNet-B2 weights not available, using ResNet34 FPN instead:", err)
    net_c, c_name = smp.FPN("resnet34", encoder_weights="imagenet", in_channels=3, classes=4), "C_resnet34_fpn"
model_c, best_c = train_model(c_name, Normalized(net_c), C_EPOCHS, 3e-4, "FPN decoder, ImageNet encoder")

### Comparing the three models (same validation split, best epoch of each)

In [ ]:
import matplotlib.pyplot as plt
compare = pd.DataFrame([best_a, best_b, best_c], index=["A scratch U-Net", "B ResNet34 U-Net", "C " + c_name])
compare = compare[["epoch", "train_loss", "val_loss", "val_dice", "val_accuracy", "val_f1"]]
print(compare.round(4))
compare.to_csv("/kaggle/working/m3_model_comparison.csv")
compare[["val_dice", "val_accuracy", "val_f1"]].plot(kind="bar", figsize=(8, 4), rot=0, ylim=(0, 1))
plt.title("Validation metrics of the three models"); plt.tight_layout()
plt.savefig("/kaggle/working/m3_model_comparison.png", dpi=150); plt.show()

### Ensemble, threshold and minimum area tuning on validation

Probabilities from hflip test time augmentation are averaged over the chosen models. For each class I pick a
threshold and a minimum mask area that give the best competition style score on the validation split.

In [ ]:
TAUS = [0.3, 0.4, 0.5, 0.6, 0.7]
MIN_AREAS = [0, 100, 250, 500, 1000, 1500, 2000, 3000, 4000, 6000]
models = {"A": model_a, "B": model_b, "C": model_c}
COMBOS = {"B": ["B"], "C": ["C"], "B+C": ["B", "C"], "A+B+C": ["A", "B", "C"]}


def tta_probs(model, x):
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=amp_on):
        p = torch.sigmoid(model(x)).float()
        p_flip = torch.sigmoid(model(torch.flip(x, dims=[3]))).float()
    return (p + torch.flip(p_flip, dims=[3])) / 2


for m in models.values():
    m.eval()
stats = {k: {"pc": [], "ic": []} for k in COMBOS}
gt_count = []
for x, y in m3_val_loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    probs = {k: tta_probs(m, x) for k, m in models.items()}
    for combo, members in COMBOS.items():
        p = sum(probs[k] for k in members) / len(members)
        pc, ic = [], []
        for t in TAUS:
            pm = p >= t
            pc.append(pm.sum((2, 3)))
            ic.append((pm & (y > 0.5)).sum((2, 3)))
        stats[combo]["pc"].append(torch.stack(pc, -1).cpu())
        stats[combo]["ic"].append(torch.stack(ic, -1).cpu())
    gt_count.append(y.sum((2, 3)).cpu())
gt_count = torch.cat(gt_count).numpy()
positive = gt_count > 0
n_pos, n_empty = positive.sum(), (~positive).sum()


def tune(pred_count, inter_count):
    cfg, total = {}, 0.0
    for c, name in enumerate(CLASSES):
        options = []
        for ti, tau in enumerate(TAUS):
            for area in MIN_AREAS:
                keep = pred_count[:, c, ti] >= max(area, 1)
                pos = positive[:, c]
                dice = np.where(keep, 2 * inter_count[:, c, ti] / np.maximum(pred_count[:, c, ti] + gt_count[:, c], 1), 0.0)
                score = 0.7 * dice[pos].sum() / n_pos + 0.3 * (~keep[~pos]).sum() / n_empty
                options.append((score, tau, area))
        cfg[name] = max(options)
        total += cfg[name][0]
    return total, cfg


combo_scores = {}
for combo in COMBOS:
    pcount = torch.cat(stats[combo]["pc"]).numpy()
    icount = torch.cat(stats[combo]["ic"]).numpy()
    combo_scores[combo] = tune(pcount, icount)
    print(f"{combo}: tuned validation score {combo_scores[combo][0]:.4f}")
best_combo = max(combo_scores, key=lambda k: combo_scores[k][0])
best_cfg = combo_scores[best_combo][1]
print("best combo:", best_combo)
for name in CLASSES:
    print(name, "threshold", best_cfg[name][1], "min_area", best_cfg[name][2])

summary_run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_comparison", mode=WANDB_MODE, reinit=True)
table = wandb.Table(columns=["model", "best_epoch", "val_loss", "val_dice", "val_accuracy", "val_f1"])
for label, row in compare.iterrows():
    table.add_data(label, int(row["epoch"]), row["val_loss"], row["val_dice"], row["val_accuracy"], row["val_f1"])
wandb.log({"model_comparison": table,
           **{f"tuned_score_{k}": v[0] for k, v in combo_scores.items()}, "best_combo": best_combo})
summary_run.finish()

### Final submission from the best combination

In [ ]:
test_ids = sample.image_id.drop_duplicates().tolist()


class TestDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image = cv2.imread(str(find_image(TEST_DIR, self.ids[idx])), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return torch.from_numpy(image.transpose(2, 0, 1).copy()), self.ids[idx]


members = COMBOS[best_combo]
test_loader3 = DataLoader(TestDataset(test_ids), batch_size=16, shuffle=False, num_workers=4)
prediction = {}
for x, ids in test_loader3:
    x = x.to(DEVICE)
    p = (sum(tta_probs(models[k], x) for k in members) / len(members)).cpu().numpy()
    for i, image_id in enumerate(ids):
        for c, name in enumerate(CLASSES):
            _, tau, area = best_cfg[name]
            m = (p[i, c] >= tau).astype(np.uint8)
            prediction[(image_id, name)] = rle_encode(m) if m.sum() >= max(area, 1) else ""

submission = sample.copy()
submission["mask_rle"] = [prediction[(i, c)] for i, c in zip(submission.image_id, submission.defect_type)]
submission["mask_rle"] = submission["mask_rle"].replace("", " ")
assert len(submission) == len(sample) and submission.ID.equals(sample.ID)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("non-empty predictions per class:")
print(submission[submission.mask_rle != " "].defect_type.value_counts())
print("Saved /kaggle/working/submission.csv using", best_combo)